# Word and Sentence Embeddings


## Feed-forward neural networks

<img class="lecture-figure figure-48" src="../img/mlp.svg" alt="A feed-forward neural network with an input layer, hidden layers and an output layer">

The network needs numerical inputs. **What numbers should represent a word?**

A feed-forward neural network takes numerical inputs. The representation of a word determines which properties of that word are available to the network.


## Representing words

A representation maps each word to a vector.

What should the vectors for **apple**, **orange** and **rabbit** look like?

We want a model to distinguish words and make use of similarities between them.

## One-hot word representations

For the vocabulary $V=\{\text{apple},\text{orange},\text{rabbit}\}$:

$$\begin{aligned}
f(\text{apple})&=(1,0,0)\\
f(\text{orange})&=(0,1,0)\\
f(\text{rabbit})&=(0,0,1)
\end{aligned}$$

A **one-hot vector** has one coordinate per vocabulary item.

## One-hot vectors are orthogonal

<img class="lecture-figure figure-45" src="../img/sparse_binary.svg" alt="The original three-dimensional visualization: apple, orange and rabbit lie on three perpendicular unit axes.">

Every pair of different words has dot product zero. The representation gives **apple–orange** no more similarity than **apple–rabbit**.

One-hot vectors encode word identity exactly, but their geometry provides no graded similarity. Sparse storage can represent a one-hot vector by its nonzero entry, without storing all the zeros.


## Cosine similarity

Cosine compares vector directions:

$$\cos(u,v)=\frac{u\cdot v}{\lVert u\rVert\,\lVert v\rVert}$$

Same direction: **1**. Orthogonal: **0**. Opposite direction: **−1**.

What happens when vectors can point between the axes?

Cosine similarity is undefined for a zero vector. SciPy's cosine distance is $1-\cos(u,v)$, so a smaller distance corresponds to a higher similarity.


## Cosine similarity

<div class="cosine-quiz">
<div><img class="lecture-figure figure-44" src="../img/word2vec_2027/cosine_quiz.svg" alt="Query q=(1,1); A=(3,3), B=(1,0.4), C=(5,-1), D=(-1,-1)."></div>
<div>
<p><b>Which vector has the highest cosine similarity to <i>q</i>?</b></p>
<p class="fragment"><b>A:</b> same direction as <i>q</i>, so cosine = 1.</p>
<p class="fragment">If we multiply <b>B</b> by 10, does the answer change?</p>
<p class="fragment"><b>No.</b> Positive scaling preserves direction.</p>
</div>
</div>

The cosine similarities to $q$ are:

| Vector | Cosine similarity |
| --- | ---: |
| A | 1 |
| B | 0.919145 |
| C | 0.554700 |
| D | −1 |

B has the closest endpoint to $q$ by Euclidean distance, while C has the largest norm. A has the highest cosine similarity because it points in the same direction as $q$. Multiplying B by ten changes its norm and endpoint distance but preserves its cosine similarity.


## Dense word embeddings

<img class="lecture-figure figure-42" src="../img/dense_continuous.svg" alt="The original two-dimensional illustration: apple and orange point in similar directions, while rabbit points in a different direction.">

A **dense word embedding** gives each word a short vector of real-valued features.

**How could we learn a useful geometry like this from text?**

The coordinates in this figure are illustrative. A dense representation can share a feature across many words, whereas a one-hot coordinate identifies a single word. Different training algorithms can learn dense representations.


## The distributional hypothesis

Words that occur in similar contexts often have related meanings.

> You shall know a word by the company it keeps.

Firth (1957)

We can turn this idea into a representation by **counting nearby words**.

Context can capture syntactic as well as semantic behaviour. Corpus choice and social patterns shape the resulting similarities, so distributional similarity is not a complete account of meaning.


## Counting context words

<div class="lecture-columns">
<div>
<p>Peel the <b>apple</b>.<br>Slice the <b>apple</b>.</p>
<p>Peel the <b>orange</b>.<br>Slice the <b>orange</b>.</p>
<p>Feed the <b>rabbit</b>.</p>
</div>
<div>
<p><b>Selected context counts</b></p>
<table><thead><tr><th>Word</th><th>peel</th><th>slice</th><th>feed</th></tr></thead>
<tbody><tr><td>apple</td><td>1</td><td>1</td><td>0</td></tr>
<tr><td>orange</td><td>1</td><td>1</td><td>0</td></tr>
<tr><td>rabbit</td><td>0</td><td>0</td><td>1</td></tr></tbody></table>
</div>
</div>

A row is a word vector. Its coordinates count words within a context window.

**Counting rule.** The example uses lowercase words with punctuation removed and a symmetric context window of radius two within each sentence. Each row represents a word, and each column represents a context word.

**Displayed features.** The table shows the context columns **peel**, **slice** and **feed**. The full vocabulary also includes **apple**, **orange**, **rabbit** and **the**. Across the corpus, apple and orange each have two occurrences of **the** in their contexts, while rabbit has one. Apple and orange have identical rows in the full context-count matrix too.


## Similar contexts

**Apple** and **orange** never occur together in this corpus.

Does that prevent their context-count vectors from being similar?

**A.** Yes, they must occur near each other.  
**B.** No, they can share the same context words.

**B.** Both occur with **peel** and **slice**. Similarity can come from shared contexts, without direct co-occurrence.

The displayed apple and orange rows have cosine similarity one. Their similarity comes from sharing context words. Two words can therefore have similar context-count vectors even if they never occur together.


## Counts and TF–IDF for documents

We can also represent a **document** by counting its words.

**Term frequency–inverse document frequency (TF–IDF)** downweights words found in many documents.

The features still correspond to vocabulary items: **apple** and **orange** occupy separate coordinates.

Can we give a model features that these related words can share?

A word–context matrix has one row per word, while a document–term matrix has one row per document.

TF–IDF is a strong baseline for document classification and retrieval. Documents can have similar TF–IDF vectors when they share words, but separate coordinates for apple and orange do not directly encode their related meanings. The smoothed scikit-learn formula and row normalization are explained in the [optional technical notes](dl-representations_notes.ipynb).


## Why use dense vectors?

For example, **50,000 context features** can become **300 learned features**.

- Related words can share features, helping a model generalize across words.
- A smaller input dimension makes neural models easier to work with.

Counts and TF–IDF remain useful baselines, especially when exact words matter.

**Feature sharing.** Dense vectors let related words share features. The dimensions 50,000 and 300 illustrate a possible reduction. A smaller input dimension also reduces the number of input weights in a fully connected layer.

**Trade-offs.** Sparse counts can be stored efficiently too. Compressing context patterns may lose useful information, so the choice of representation depends on the task. Weighting and compressing context counts provides another route to dense vectors, developed in the [optional technical notes](dl-representations_notes.ipynb).


## Learning a representation through prediction

The feed-forward network can learn features that help it predict a word from its context.

> Peel the **_____**.

The original text, “Peel the **apple**”, supplies the word to predict. **The corpus supplies the training targets.**

The network learns a representation that helps with word prediction. Apple and orange can both fit this context, but each occurrence in the corpus supplies one observed word as the training target. Learning over many contexts can make their representations similar. This is self-supervised learning: the training targets come from the text itself, as in language modelling.


## Continuous bag-of-words (CBOW)

Average the vectors of the context words and predict the missing word.

$$\underbrace{\frac{f(\text{peel})+f(\text{the})}{2}}_{\text{context representation}}\quad\longrightarrow\quad\text{predict apple}$$

Training these predictions also learns the word vectors being averaged.

CBOW learns word vectors by predicting a word from pooled context vectors. The example uses mean pooling. The context window normally includes words on both sides when available, but apple is at the end of this sentence. CBOW variants can pool by summing or averaging the context vectors. Source: [Mikolov et al. (2013), Efficient Estimation of Word Representations in Vector Space](https://arxiv.org/abs/1301.3781).


## Word2vec: CBOW and skip-gram

<img class="lecture-figure figure-40" src="../img/word2vec_2027/cbow_skipgram.svg" alt="CBOW averages the context vectors for peel and the to predict apple. Skip-gram uses apple to predict peel and the separately.">

**Word2vec** uses a **shallow neural network** to learn word vectors.

- **Training:** word–context prediction using **stochastic gradient descent (SGD)**.
- **Initialization:** small random word vectors.


Word2vec has one hidden projection layer and no nonlinear hidden activation. In CBOW it represents the pooled context, while in skip-gram it represents the input word. Both architectures learn static embeddings: one input vector per vocabulary item.

The co-occurrences counted earlier now supply prediction examples. The following network diagram shows skip-gram. Source: [Mikolov et al. (2013)](https://arxiv.org/abs/1301.3781).


## Skip-gram architecture

<img class="lecture-figure figure-44" src="../img/word2vec_2027/network.svg" alt="A one-hot input selects apple. Input-to-hidden weights produce a small linear embedding layer. Hidden-to-output weights score context words, with peel highlighted as an observed context.">

The **input weights** store the word vectors. A one-hot input selects one vector, and the **output weights** map it to context predictions.


**Input weights.** For vocabulary size $|V|$ and embedding dimension $d$, $W_{\mathrm{in}}\in\mathbb{R}^{|V|\times d}$ has one row per word. Multiplying a one-hot row vector by this matrix selects the corresponding word vector.

**Output weights.** $W_{\mathrm{out}}\in\mathbb{R}^{d\times |V|}$ maps that vector to scores over context words. Softmax converts the scores to probabilities.

Both sets of weights are trainable parameters. A common downstream choice keeps the input vectors as word embeddings. The diagram displays selected vocabulary items and three embedding coordinates.


## Initialization and training

<img class="lecture-figure figure-43" src="../img/word2vec_2027/training.svg" alt="Word vectors start as small random numbers. A word and its vector feed a context prediction, which is compared with the observed context. Backpropagation and stochastic gradient descent update the network weights.">

Input word vectors start as **small random values**. The original implementation initializes output weights to **zero**.

**SGD** updates both sets of weights to improve context prediction.


**Initialization.** The random input weights initially carry no semantic information. The zero output weights are a choice made by the original word2vec C implementation. Other implementations can choose different initializations.

**Training.** The model predicts context words from corpus examples. Backpropagation computes gradients of the prediction loss, and stochastic gradient descent updates the weights, including the word vectors.

For large vocabularies, negative sampling reduces computation by training a classifier on observed and sampled word–context pairs. The [optional technical notes](dl-representations_notes.ipynb) explain negative sampling and a worked update. Sources: [original implementation](https://github.com/tmikolov/word2vec/blob/master/word2vec.c) and [Mikolov et al. (2013)](https://arxiv.org/abs/1310.4546).


## Learned word embeddings

<img class="lecture-figure figure-48" src="../img/word_representations.svg" alt="The original projection of a learned word embedding space, with clusters of words that share syntactic or semantic behaviour.">

Words with similar contexts can acquire similar vectors. **We can reuse these vectors as inputs to other models.**

The figure is a two-dimensional t-distributed stochastic neighbor embedding (t-SNE) projection of higher-dimensional vectors. Projection can distort distances, so similarities in the plot may differ from those in the original space. Word vectors often reflect syntactic as well as semantic patterns. Different word senses and biases in the training corpus remain limitations.


## How counts and prediction are connected

Both use evidence about **which words occur near one another**.

- **Count-based methods** can produce dense vectors by fitting or compressing co-occurrence statistics. **Global Vectors (GloVe)** is one example.
- **Word2vec** learns dense vectors by optimizing context predictions.

The shared evidence connects them. Their different objectives can produce different vectors.

Count-based and prediction-based describe how vectors are learned. Both approaches can produce dense vectors. GloVe fits weighted log co-occurrence counts using learned vectors and bias terms. Some prediction objectives also have a matrix-factorization interpretation under stated assumptions. The [optional technical notes](dl-representations_notes.ipynb) develop this connection and show how weighting and compressing counts can produce dense vectors.

Sources: [Pennington et al. (2014)](https://aclanthology.org/D14-1162/), [Levy & Goldberg (2014)](https://papers.nips.cc/paper_files/paper/2014/file/b78666971ceae55a8e87efb7cbfd9ad4-Paper.pdf), and [Levy et al. (2015)](https://aclanthology.org/Q15-1016/).


## Representing a whole sentence

We now have one vector per word. A classifier may need one vector for the **whole sentence**.

**Mean pooling of static word embeddings** averages its word vectors:

$$f(s)=\frac{1}{n}\sum_{i=1}^{n}f(w_i)$$

The result is a **sentence embedding** with the same dimension as the word vectors.

Static means that the same word vector is used in every sentence. Mean pooling has no trainable parameters of its own. It can combine vectors learned with CBOW, skip-gram, GloVe or other methods.


## Averaging during training and after training

**CBOW training:** average a context window to predict a word. Prediction errors update the word vectors.

**Sentence mean pooling:** average already learned vectors across a complete sentence.

The averaging operation is the same. What we represent and what we train are different.

- **CBOW training:** for “peel the ___”, average the vectors for **peel** and **the** to predict **apple**.
- **Sentence mean pooling:** for “peel the apple”, average all three word vectors to represent the sentence.

Sentence pooling can use vectors learned by any of the word-embedding methods introduced here.


## Who chases whom?

<div class="sentence-pair"><p><b>dogs</b> chase cats</p><p><b>cats</b> chase dogs</p></div>

With the same static word vectors, will these sentences have **the same mean-pooled vector or different vectors?**

**The same vector:**

$$\frac{f(\text{dogs})+f(\text{chase})+f(\text{cats})}{3}
=\frac{f(\text{cats})+f(\text{chase})+f(\text{dogs})}{3}$$

The meaning changes, but the average loses word order.

Reordering words leaves their vector sum unchanged. Any two sentences with the same word tokens and frequencies therefore have the same mean-pooled vector, even when the words play different grammatical roles.


In [ ]:
import numpy as np

word_vectors = {
    "dogs": [1., 0.], "chase": [0., 1.], "cats": [.8, .2]
}

def mean_pool(sentence):
    return np.mean([word_vectors[w] for w in sentence.split()], axis=0)

## Mean pooling in code

Each word has a vector. Averaging ignores the order in which we add them.

In [ ]:
for sentence in ("dogs chase cats", "cats chase dogs"):
    print(sentence, mean_pool(sentence))

dogs chase cats [0.6 0.4]
cats chase dogs [0.6 0.4]


The setup cell defines illustrative word vectors and the `mean_pool` function. To run the example, execute that cell before the loop. Both sentences produce `[0.6, 0.4]` because NumPy averages the same three vectors.


## Mean pooling in the library

The **Sentence Transformers library** can average static GloVe word embeddings:

```python
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/average_word_embeddings_glove.6B.300d"
)
vectors = model.encode(["dogs chase cats", "cats chase dogs"])
```

This model contains a word-vector lookup and mean pooling. Each sentence gets a 300-dimensional vector.

`SentenceTransformer` is the library interface used to load this model. The model itself uses static GloVe vectors and mean pooling, with no Transformer layers. Running the example requires the `sentence-transformers` package and a model download. See the [model documentation](https://huggingface.co/sentence-transformers/average_word_embeddings_glove.6B.300d).


## What is still missing?

The word **bank** has the same static vector in “river bank” and “bank loan”.

Mean pooling also loses **word order**, as “dogs chase cats” showed.

Next: models that use the sequence of words and build representations in context.

The word vector for bank is unchanged, but the mean-pooled sentence vectors can still differ because the other words contribute their own vectors. Contextual representations allow the vector for bank itself to depend on the surrounding words.


## Summary

- One-hot vectors distinguish words but give no graded similarity.
- Context counts reveal shared usage. Dense vectors can encode useful patterns in fewer features.
- A prediction network learns word embeddings as part of its weights.
- Mean pooling gives a sentence vector, while losing word order.

## Further reading

- Jurafsky & Martin: [Embeddings](https://web.stanford.edu/~jurafsky/slp3/5.pdf)
- Mikolov et al. (2013): [Word2vec architectures](https://arxiv.org/abs/1301.3781)
- Pennington et al. (2014): [GloVe](https://aclanthology.org/D14-1162/)
- [Optional technical notes](dl-representations_notes.ipynb): count weighting, compression and training details